# Data preparation and model design

This notebook reads and verifies the committed prepared splits. It summarizes the audit, split balance, group leakage checks, and current neural model designs. It does not rebuild data or train models.

Run from the repository root or this `notebooks/` directory after installing the project dependencies. To intentionally regenerate the prepared data, use `uv run python -m scripts.build_dataset` in a terminal and review the resulting dataset and report changes.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "app").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "app").is_dir():
    raise RuntimeError("Open this notebook from the repository root or notebooks directory.")
sys.path.insert(0, str(ROOT))

import json
import pandas as pd
from IPython.display import display

from app.config import PREPARED_DATA_DIR, STYLES, TONES
from app.data import load_splits

frames, manifest = load_splits(PREPARED_DATA_DIR)
print(f"Verified prepared data: {manifest['clean_rows']} rows")
print(f"Excluded conflicting rows: {manifest['conflicting_rows_removed']}")
print(f"Split seed: {manifest['seed']}")

In [ ]:
split_summary = pd.DataFrame(
    [
        {
            "split": name,
            "rows": len(frame),
            "styles": frame["style"].nunique(),
            "tones": frame["tone"].nunique(),
            "groups": frame["group_id"].nunique(),
        }
        for name, frame in frames.items()
    ]
)
display(split_summary)

label_rows = []
for split_name, frame in frames.items():
    for label in STYLES:
        label_rows.append({"split": split_name, "task": "style", "label": label, "rows": int(frame["style"].eq(label).sum())})
    for label in TONES:
        label_rows.append({"split": split_name, "task": "tone", "label": label, "rows": int(frame["tone"].eq(label).sum())})
display(pd.DataFrame(label_rows).pivot(index=["task", "label"], columns="split", values="rows"))

In [ ]:
overlap_rows = []
split_names = list(frames)
for left_index, left_name in enumerate(split_names):
    left_groups = set(frames[left_name]["group_id"])
    for right_name in split_names[left_index + 1 :]:
        right_groups = set(frames[right_name]["group_id"])
        overlap_rows.append(
            {
                "split_pair": f"{left_name} / {right_name}",
                "shared_groups": len(left_groups & right_groups),
            }
        )
overlap = pd.DataFrame(overlap_rows)
display(overlap)
assert overlap["shared_groups"].eq(0).all()

## Model designs and evaluation protocol

All architectures predict style and tone through separate output heads. Training selects checkpoints using validation mean macro F1 and fits calibration temperatures on validation predictions. The test split is held out for final evaluation and must not guide architecture or checkpoint selection.

In [ ]:
architectures = pd.DataFrame(
    [
        {"model": "TextCNN", "representation": "Trainable token embeddings", "design": "Convolution windows 3/4/5 with padding-safe pooling; separate style and tone heads"},
        {"model": "BiLSTM", "representation": "Trainable token embeddings", "design": "Packed bidirectional sequences with masked attention; separate style and tone heads"},
        {"model": "DistilBERT", "representation": "Fine-tuned English cased encoder", "design": "Saved tokenizer and attention masks; separate style and tone heads"},
    ]
)
display(architectures)